<img align="right" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>
<h1 align="left">Biigle → YOLO Dataset Converter</h1>
<h4 align="left">Written by the KSO Team</h4>

This notebook converts **Biigle CSV** annotations into a training-ready **YOLO dataset** with `train/valid/test` splits and a `data.yaml` file.

---

### Quick Start

1. Export a **CSV** report from your project in [Biigle.de](https://biigle.de)
2. Create a directory for your Biigle data (e.g. `/scratch/.../my_biigle_data/`)
3. Upload the CSV file to that directory
4. Upload your raw images to a subdirectory called `raw_frames/`
5. Choose a path where this notebook will create your YOLO dataset
6. Edit **Phase 1** configuration with your paths
7. Run all cells top to bottom → use output with `Train_Models.ipynb`

---

### Features

<div align="left">

| | |
|:--|:--|
| **Image-level splits** | No data leakage—splits by image, not annotation |
| **Semi-stratified** | Each class guaranteed in train; covered in val/test where possible |
| **Offline augmentation** | Optional flips/rotations on train split only |
| **Reproducible** | Seeded random operations |

</div>

---
## Phase 1: Configuration

Edit the fields below to match your setup, then run the cell.

In [ ]:
# =============================================================================
# PHASE 1: CONFIGURATION
# =============================================================================

from pathlib import Path

# ----- INPUT -----
BIIGLE_DIR = Path("<path-to-biigle-data>")  # <-- Your CSV and images directory

BIIGLE_CSV_PATH = BIIGLE_DIR / "<your-export>.csv"  # <-- Your Biigle CSV
IMAGES_ROOT = BIIGLE_DIR / "raw_frames"  # Filenames must match CSV

# ----- OUTPUT -----
OUTPUT_ROOT = Path("<path-to-datasets-dir>")  # <-- e.g. "/scratch/.../datasets"
DATASET_NAME = "<your-dataset-name>"  # <-- Name your dataset. e.g. "fish_dataset_1"
DATASET_DIR = OUTPUT_ROOT / DATASET_NAME  # Where the YOLO dataset will be created

# ----- SPLIT RATIOS -----
# Standard: 70% train, 20% validation, 10% test
TRAIN_RATIO = 0.7
VAL_RATIO = 0.2
TEST_RATIO = 0.1

# ----- AUGMENTATION (train split only) -----
# Set AUGMENT_TRAIN = False to disable
AUGMENT_TRAIN = True
AUGMENT_FACTOR = 0.5  # 0.5 = +50% images, 1.0 = double
AUGMENT_OPS = [
    "hflip",
    "vflip",
    "rot180",
]  # Available: hflip, vflip, rot90, rot180, rot270

# ----- CONSTANTS -----
RANDOM_SEED = 42
VALID_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

# ----- VALIDATION -----
assert (
    abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 0.01
), f"Split ratios must sum to 1.0, got {TRAIN_RATIO + VAL_RATIO + TEST_RATIO:.2f}"
assert BIIGLE_CSV_PATH.is_file(), f"CSV file not found: {BIIGLE_CSV_PATH}"
assert IMAGES_ROOT.is_dir(), f"Images directory not found: {IMAGES_ROOT}"

if DATASET_DIR.exists() and any(DATASET_DIR.iterdir()):
    print(f"Warning: Output directory not empty: {DATASET_DIR}")
    print("   Consider changing DATASET_DIR to avoid overwriting.")
else:
    DATASET_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  CSV:       {BIIGLE_CSV_PATH}")
print(f"  Images:    {IMAGES_ROOT}")
print(f"  Output:    {DATASET_DIR}")
print(f"  Splits:    {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")
print(f"  Augment:   {AUGMENT_TRAIN} (factor={AUGMENT_FACTOR})")

---
## Phase 2: Load & Parse Biigle CSV

Load annotations, filter to supported shapes, and convert to YOLO format.

In [ ]:
# =============================================================================
# PHASE 2: LOAD & PARSE BIIGLE CSV
# =============================================================================

import json
import pandas as pd

df = pd.read_csv(BIIGLE_CSV_PATH)
print(f"Loaded {len(df)} annotations from CSV")

# Validate columns
required = ["filename", "label_name", "shape_name", "points", "attributes"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"CSV missing required columns: {missing}")

# Filter to Rectangle/Polygon (both converted to axis-aligned bounding boxes)
supported = ["Rectangle", "Polygon"]
df = df[df["shape_name"].isin(supported)].reset_index(drop=True)
print(f"Supported annotations (Rectangle/Polygon): {len(df)}")

# Build class mapping
class_names = sorted(df["label_name"].unique())
df["class_id"] = df["label_name"].map({name: i for i, name in enumerate(class_names)})

print(f"\nDetected {len(class_names)} classes:")
for cid, name in enumerate(class_names):
    print(f"  {cid}: {name} ({(df['class_id'] == cid).sum()} annotations)")

# Parse image dimensions from attributes
attrs = df["attributes"].apply(json.loads)
df["img_w"] = [d.get("width") for d in attrs]
df["img_h"] = [d.get("height") for d in attrs]
df = df.dropna(subset=["img_w", "img_h"]).reset_index(drop=True)


# Convert to YOLO format
def bbox_from_points(points_str):
    """Convert Biigle point coordinates to axis-aligned bounding box."""
    pts = json.loads(points_str)
    xs, ys = pts[0::2], pts[1::2]
    return min(xs), min(ys), max(xs), max(ys)


df[["xmin", "ymin", "xmax", "ymax"]] = df["points"].apply(
    lambda s: pd.Series(bbox_from_points(s))
)

df["x_center"] = ((df["xmin"] + df["xmax"]) / 2) / df["img_w"]
df["y_center"] = ((df["ymin"] + df["ymax"]) / 2) / df["img_h"]
df["w_norm"] = (df["xmax"] - df["xmin"]) / df["img_w"]
df["h_norm"] = (df["ymax"] - df["ymin"]) / df["img_h"]

# Drop degenerate boxes (zero/negative area or out-of-bounds)
n_before = len(df)
df = df[(df["w_norm"] > 0) & (df["h_norm"] > 0)].reset_index(drop=True)
df[["x_center", "y_center", "w_norm", "h_norm"]] = df[
    ["x_center", "y_center", "w_norm", "h_norm"]
].clip(0, 1)
if n_before - len(df) > 0:
    print(f"  Dropped {n_before - len(df)} degenerate annotations (zero-area boxes)")

print(f"\nConverted {len(df)} annotations to YOLO format")

---
## Phase 3: Split & Build Dataset

Create train/valid/test splits and write images + labels to YOLO directory structure.

In [ ]:
# =============================================================================
# PHASE 3: SPLIT & BUILD YOLO DATASET
# =============================================================================

from collections import defaultdict
import random
import shutil

# Index source images
image_index = {
    p.name: p
    for p in IMAGES_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in VALID_EXTS
}

# Check for CSV filenames that don't exist on disk
csv_filenames = set(df["filename"].unique())
unmatched = csv_filenames - set(image_index.keys())
if unmatched:
    print(
        f"Warning: {len(unmatched)} filenames in CSV not found in {IMAGES_ROOT.name}/"
    )
    for fn in sorted(unmatched)[:5]:
        print(f"  - {fn}")
    if len(unmatched) > 5:
        print(f"  ... and {len(unmatched) - 5} more")
    print()

# Create splits
rng = random.Random(RANDOM_SEED)
img_to_labels = defaultdict(set)
for row in df.itertuples():
    img_to_labels[row.filename].add(int(row.class_id))

images = list(img_to_labels.keys())
n_images = len(images)
n_train = int(round(TRAIN_RATIO * n_images))
n_val = int(round(VAL_RATIO * n_images))

split_for_image = {}
split_counts = {"train": 0, "val": 0, "test": 0}


def assign(img, split):
    if img not in split_for_image:
        split_for_image[img] = split
        split_counts[split] += 1
        return True
    return split_for_image[img] == split


# Ensure class coverage: at least one image per class in each split
for cls in sorted(df["class_id"].unique()):
    imgs = [img for img, labels in img_to_labels.items() if cls in labels]
    rng.shuffle(imgs)
    if imgs:
        assign(imgs[0], "train")
    if len(imgs) >= 2:
        for img in imgs[1:]:
            if assign(img, "val"):
                break
    if len(imgs) >= 3:
        for img in imgs[2:]:
            if assign(img, "test"):
                break

# Fill remaining images to match target ratios
remaining = [img for img in images if img not in split_for_image]
rng.shuffle(remaining)
for img in remaining:
    needs = {
        "train": n_train - split_counts["train"],
        "val": n_val - split_counts["val"],
        "test": (n_images - n_train - n_val) - split_counts["test"],
    }
    max_need = max(needs.values())
    if max_need <= 0:
        assign(img, "train")
    else:
        assign(img, rng.choice([s for s, n in needs.items() if n == max_need]))

# Build dataset directories
SPLIT_MAP = {"train": "train", "val": "valid", "test": "test"}
for folder in SPLIT_MAP.values():
    (DATASET_DIR / folder / "images").mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / folder / "labels").mkdir(parents=True, exist_ok=True)

grouped = {fn: g for fn, g in df.groupby("filename")}
stats = {f: {"images": 0, "annotations": 0} for f in SPLIT_MAP.values()}

for filename, split in split_for_image.items():
    folder = SPLIT_MAP[split]
    src_img = image_index.get(filename)
    if src_img is None:
        continue

    shutil.copy2(src_img, DATASET_DIR / folder / "images" / filename)

    label_lines = []
    if filename in grouped:
        for row in grouped[filename].itertuples():
            label_lines.append(
                f"{int(row.class_id)} {row.x_center:.6f} {row.y_center:.6f} "
                f"{row.w_norm:.6f} {row.h_norm:.6f}"
            )

    (DATASET_DIR / folder / "labels" / f"{Path(filename).stem}.txt").write_text(
        "\n".join(label_lines)
    )

    stats[folder]["images"] += 1
    stats[folder]["annotations"] += len(label_lines)

print(f"Dataset created from {n_images} images:")
for folder, s in stats.items():
    print(f"  {folder}: {s['images']} images, {s['annotations']} annotations")

# Per-class split breakdown
df["split"] = df["filename"].map(split_for_image)
print(f"\nPer-class distribution:")
header = f"  {'Class':<25s} {'train':>6s} {'valid':>6s} {'test':>6s}"
print(header)
print("  " + "-" * (len(header) - 2))
for cid, name in enumerate(class_names):
    counts = df[df["class_id"] == cid]["split"].value_counts()
    print(
        f"  {name:<25s} {counts.get('train', 0):>6d} "
        f"{counts.get('val', 0):>6d} {counts.get('test', 0):>6d}"
    )

---
## Phase 4: Augmentation (Optional)

Apply geometric augmentations to training images only. Adjust `AUGMENT_FACTOR` in Phase 1 to control intensity.

In [ ]:
# =============================================================================
# PHASE 4: OFFLINE AUGMENTATION (TRAIN ONLY)
# =============================================================================

from PIL import Image

if not AUGMENT_TRAIN or AUGMENT_FACTOR <= 0:
    print("Augmentation disabled.")
else:
    train_img_dir = DATASET_DIR / "train" / "images"
    train_lbl_dir = DATASET_DIR / "train" / "labels"

    train_imgs = sorted(
        p for p in train_img_dir.iterdir() if p.suffix.lower() in VALID_EXTS
    )
    n_original = len(train_imgs)
    n_target = int(round(n_original * AUGMENT_FACTOR))

    if n_target == 0:
        print("No augmentation needed.")
    else:
        rng = random.Random(RANDOM_SEED + 1)
        selected = rng.sample(train_imgs, k=min(n_target, n_original))

        def transform_bbox(xc, yc, w, h, op):
            """Transform YOLO bbox coordinates for an augmentation operation."""
            if op == "hflip":
                return 1 - xc, yc, w, h
            if op == "vflip":
                return xc, 1 - yc, w, h
            if op == "rot180":
                return 1 - xc, 1 - yc, w, h
            if op == "rot90":
                return yc, 1 - xc, h, w
            if op == "rot270":
                return 1 - yc, xc, h, w
            raise ValueError(f"Unknown op: {op}")

        def apply_image_op(img, op):
            """Apply augmentation operation to a PIL Image."""
            if op == "hflip":
                return img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            if op == "vflip":
                return img.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
            if op == "rot180":
                return img.rotate(180, expand=True)
            if op == "rot90":
                return img.rotate(90, expand=True)
            if op == "rot270":
                return img.rotate(-90, expand=True)
            raise ValueError(f"Unknown op: {op}")

        created = 0
        for img_path in selected:
            label_path = train_lbl_dir / f"{img_path.stem}.txt"
            if not label_path.exists():
                continue

            raw = label_path.read_text().strip()
            if not raw:
                continue

            op = rng.choice(AUGMENT_OPS)
            new_labels = []
            for line in raw.splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls = parts[0]
                xc, yc, w, h = map(float, parts[1:5])
                xc, yc, w, h = transform_bbox(xc, yc, w, h, op)
                xc, yc = max(0, min(1, xc)), max(0, min(1, yc))
                w, h = max(0, min(1, w)), max(0, min(1, h))
                new_labels.append(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

            if not new_labels:
                continue

            img = Image.open(img_path)
            img_aug = apply_image_op(img, op)

            suffix = f"_aug_{op}"
            img_aug.save(train_img_dir / f"{img_path.stem}{suffix}{img_path.suffix}")
            (train_lbl_dir / f"{img_path.stem}{suffix}.txt").write_text(
                "\n".join(new_labels)
            )
            created += 1

        print(f"Created {created} augmented images")
        print(f"  Train set: {n_original} \u2192 {n_original + created} images")

---
## Phase 5: Write data.yaml

Generate the YOLO configuration file required for training.

In [ ]:
# =============================================================================
# PHASE 5: WRITE data.yaml
# =============================================================================

import yaml

data_yaml_path = DATASET_DIR / "data.yaml"

data_cfg = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(class_names),
    "names": class_names,
}

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print(f"Wrote {data_yaml_path}")
print("")
print(data_yaml_path.read_text())

---
## ✅ Done

Your dataset is ready in the `datasets` directory:

```
<DATASET_NAME>/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

### Next Steps

1. **Navigate to the next Notebook** `Train_Models.ipynb` and set your dataset path:
```python
data_path = Path("<path-to-your-dataset>")
```

2. **Adjust hyperparameters** (model size, epochs, batch size) and start training